# 05. Error Handling

## What you'll learn

- How to catch and handle exceptions with `try`/`except`
- Common exception types you'll encounter when building agents (`KeyError`, `ValueError`, `JSONDecodeError`)
- The full `try`/`except`/`else`/`finally` control flow
- How to raise your own exceptions for invalid states
- Retry patterns: simple loops and exponential backoff (critical for API calls)
- Basic logging to track what your agent is doing

## Prerequisites

- [01. Python Fundamentals](01_python_fundamentals.ipynb) — variables, types, control flow
- [02. Functions and Scope](02_functions_and_scope.ipynb) — defining and calling functions
- [03. Data Structures](03_data_structures.ipynb) — dicts, lists
- [04. Strings and JSON](04_strings_and_json.ipynb) — JSON parsing, string manipulation

## Why this matters for agents

AI agents live in a world of unreliable inputs. LLMs return malformed JSON. APIs time out.
Tool calls fail. Keys go missing from response dicts. An agent that can't handle errors
gracefully is an agent that crashes on the first unexpected response.

Every real agent loop has a `try`/`except` at its heart. Retry logic with backoff is how
agents survive rate limits and transient failures. Logging is how you debug what went wrong
on step 47 of a 50-step plan. This notebook gives you those foundations.

> **Further reading:**
> - [Python Errors & Exceptions (official tutorial)](https://docs.python.org/3/tutorial/errors.html) — the definitive reference
> - [Python Exceptions: An Introduction (Real Python)](https://realpython.com/python-exceptions/) — excellent walkthrough with examples
> - [Corey Schafer — Error Handling (YouTube)](https://www.youtube.com/watch?v=NIWwJbo-9_8) — try/except walkthrough
> - [Python docs — Built-in Exceptions hierarchy](https://docs.python.org/3/library/exceptions.html#exception-hierarchy) — the full exception tree
> - [Real Python — logging module](https://realpython.com/python-logging/) — production logging patterns
> - [Tenacity library docs](https://tenacity.readthedocs.io/) — the production-grade retry library
> - [Anthropic — Error Handling Best Practices](https://docs.anthropic.com/en/docs/build-with-claude/error-handling) — API error patterns relevant to agents
> - [AWS — Exponential Backoff and Jitter](https://aws.amazon.com/blogs/architecture/exponential-backoff-and-jitter/) — the definitive retry strategy article

---
## 1. `try`/`except` Basics

When Python encounters an error, it **raises an exception**. Without handling, this
crashes your program. The `try`/`except` block lets you catch exceptions and decide
what to do instead.

```
try:
    # code that might fail
except SomeException:
    # what to do when it fails
```

Let's start with a scenario you'll hit constantly: parsing an LLM response that should
be a dict but might be missing expected keys.

In [ ]:
# An LLM response we expect to contain an "action" key
llm_response = {"thought": "I should search for the answer", "observation": "none yet"}

# Without error handling, this crashes:
try:
    action = llm_response["action"]
    print(f"Action: {action}")
except KeyError:
    print("No 'action' key in LLM response — using default action.")
    action = "think"

print(f"Proceeding with action: {action}")

You can also capture the exception object using `as` to inspect the details:

In [ ]:
tool_registry = {"search": "web_search", "calculate": "calculator"}

requested_tool = "summarize"

try:
    tool_fn = tool_registry[requested_tool]
except KeyError as e:
    print(f"Unknown tool requested: {e}")
    print(f"Available tools: {list(tool_registry.keys())}")
    tool_fn = None

print(f"Resolved tool: {tool_fn}")

---
## 2. Common Exception Types

When building agents, you'll encounter a handful of exception types over and over.
Knowing them helps you write targeted `except` clauses instead of catching everything
blindly.

| Exception | When it happens | Agent context |
|-----------|----------------|---------------|
| `KeyError` | Missing dict key | LLM response missing expected field |
| `ValueError` | Wrong value for an operation | Can't convert LLM output to int |
| `TypeError` | Wrong type passed | Passing `None` where string expected |
| `json.JSONDecodeError` | Invalid JSON string | LLM returns malformed JSON |
| `IndexError` | List index out of range | Accessing tool args that don't exist |
| `ConnectionError` | Network failure | API endpoint unreachable |

In [ ]:
import json

# Simulated LLM outputs — some valid, some broken
llm_outputs = [
    '{"action": "search", "query": "Python agents"}',   # valid JSON
    '{action: search}',                                     # missing quotes — invalid
    '{"action": "calculate", "value": "not_a_number"}',  # valid JSON, bad value
]

for i, raw_output in enumerate(llm_outputs):
    print(f"\n--- Output {i + 1} ---")
    try:
        parsed = json.loads(raw_output)
        print(f"Parsed: {parsed}")
        
        # Try to use the value field as a number
        if "value" in parsed:
            number = int(parsed["value"])
            print(f"Numeric value: {number}")
            
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e.msg} at position {e.pos}")
    except ValueError as e:
        print(f"Value conversion error: {e}")

**Important:** Order matters. Python checks `except` clauses top to bottom and runs
the first one that matches. Put more specific exceptions before more general ones.

Note that `json.JSONDecodeError` is a subclass of `ValueError`, so if you put
`except ValueError` first, it would catch JSON errors too — and you'd lose the
specific error information.

In [ ]:
# Catching multiple exception types in one clause
agent_message = {"role": "assistant", "content": None}

try:
    # content might be None, or might not have the key at all
    text = agent_message["content"].strip().lower()
except (AttributeError, KeyError) as e:
    print(f"Could not extract text: {type(e).__name__}: {e}")
    text = ""

print(f"Extracted text: '{text}'")

---
## 3. `try`/`except`/`else`/`finally`

The full syntax gives you fine-grained control:

```python
try:
    # attempt something risky
except SomeError:
    # handle the error
else:
    # runs ONLY if no exception was raised
finally:
    # ALWAYS runs, whether or not there was an exception
```

- **`else`**: Use this for code that should only run on success. Keeps the `try` block
  minimal (you should only put code that might fail in `try`).
- **`finally`**: Use this for cleanup — closing files, releasing resources, logging that
  an operation completed.

In [ ]:
import json

def process_agent_response(raw_text: str) -> dict | None:
    """Parse an agent's JSON response with full try/except/else/finally."""
    result = None
    try:
        data = json.loads(raw_text)
    except json.JSONDecodeError as e:
        print(f"  [ERROR] Failed to parse JSON: {e.msg}")
    else:
        # Only runs if parsing succeeded
        print(f"  [OK] Parsed successfully: action='{data.get('action', 'N/A')}'")
        result = data
    finally:
        # Always runs
        print(f"  [DONE] Finished processing (result={'valid' if result else 'failed'})")
    return result


# Test with valid and invalid input
print("Test 1 — valid JSON:")
process_agent_response('{"action": "search", "query": "AI agents"}')

print("\nTest 2 — invalid JSON:")
process_agent_response("This is not JSON at all")

---
## 4. Raising Exceptions

Sometimes *you* need to signal that something went wrong. Use `raise` to throw an
exception when your code detects an invalid state.

This is essential for building reliable agent tools — a tool should raise a clear
exception when given bad inputs rather than silently returning garbage.

In [ ]:
def validate_tool_definition(tool: dict) -> None:
    """Validate that a tool definition has all required fields."""
    required_fields = ["name", "description", "parameters"]
    
    for field in required_fields:
        if field not in tool:
            raise ValueError(f"Tool definition missing required field: '{field}'")
    
    if not isinstance(tool["name"], str) or not tool["name"].strip():
        raise TypeError("Tool 'name' must be a non-empty string")
    
    if not isinstance(tool["parameters"], dict):
        raise TypeError("Tool 'parameters' must be a dict")


# Valid tool
good_tool = {
    "name": "web_search",
    "description": "Search the web for information",
    "parameters": {"query": {"type": "string"}},
}

try:
    validate_tool_definition(good_tool)
    print(f"Tool '{good_tool['name']}' is valid.")
except (ValueError, TypeError) as e:
    print(f"Invalid tool: {e}")


# Invalid tool — missing 'parameters'
bad_tool = {"name": "broken", "description": "A broken tool"}

try:
    validate_tool_definition(bad_tool)
    print(f"Tool '{bad_tool['name']}' is valid.")
except (ValueError, TypeError) as e:
    print(f"Invalid tool: {e}")

You can also define **custom exception classes** for your agent system. This makes it
easy to distinguish agent-specific errors from generic Python errors:

In [ ]:
class AgentError(Exception):
    """Base exception for agent-related errors."""
    pass

class ToolNotFoundError(AgentError):
    """Raised when the agent requests a tool that doesn't exist."""
    pass

class MaxRetriesExceededError(AgentError):
    """Raised when an operation fails after all retry attempts."""
    def __init__(self, operation: str, attempts: int):
        self.operation = operation
        self.attempts = attempts
        super().__init__(f"'{operation}' failed after {attempts} attempts")


# Using custom exceptions
available_tools = ["search", "calculate"]

try:
    requested = "summarize"
    if requested not in available_tools:
        raise ToolNotFoundError(f"Tool '{requested}' not found. Available: {available_tools}")
except ToolNotFoundError as e:
    print(f"Caught ToolNotFoundError: {e}")
except AgentError as e:
    # This would catch any AgentError subclass not caught above
    print(f"Caught generic AgentError: {e}")

---
## 5. Retry Patterns

API calls fail. LLMs return malformed responses. Networks glitch. **Retrying** is one
of the most important patterns for building robust agents.

### Simple retry loop

The simplest pattern: try up to N times, break on success.

In [ ]:
import json
import random

def flaky_llm_call() -> str:
    """Simulate an LLM that sometimes returns bad JSON."""
    responses = [
        '{"action": "search", "query": "agents"}',  # good
        'Sure! Here is the answer...',                  # bad — not JSON
        '{"action": "search", query: agents}',         # bad — invalid JSON
    ]
    return random.choice(responses)


def call_with_retries(max_retries: int = 3) -> dict | None:
    """Call the LLM and retry on failure."""
    for attempt in range(1, max_retries + 1):
        raw = flaky_llm_call()
        try:
            result = json.loads(raw)
            print(f"  Attempt {attempt}: Success! -> {result}")
            return result
        except json.JSONDecodeError:
            print(f"  Attempt {attempt}: Failed to parse: '{raw[:40]}...'")
    
    print(f"  All {max_retries} attempts failed.")
    return None


random.seed(0)  # for reproducibility
print("Calling flaky LLM with retries:")
result = call_with_retries(max_retries=5)
print(f"Final result: {result}")

### Exponential backoff

When hitting rate limits or transient server errors, you want to wait between retries
and increase the wait time exponentially. This is the standard pattern for API calls:

```
wait = base_delay * (2 ** attempt)
```

So with `base_delay=1`: wait 1s, 2s, 4s, 8s...

Adding a small random **jitter** prevents many clients from retrying at exactly the
same time (the "thundering herd" problem).

In [ ]:
import time
import random

def retry_with_backoff(
    func,
    max_retries: int = 4,
    base_delay: float = 0.1,  # small for demo; use 1.0+ in production
    max_delay: float = 2.0,
) -> any:
    """Retry a function with exponential backoff and jitter."""
    for attempt in range(max_retries):
        try:
            return func()
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"  Attempt {attempt + 1}: Final failure — giving up.")
                raise
            
            delay = min(base_delay * (2 ** attempt), max_delay)
            jitter = random.uniform(0, delay * 0.5)
            total_wait = delay + jitter
            
            print(f"  Attempt {attempt + 1}: {type(e).__name__}: {e}")
            print(f"    Waiting {total_wait:.2f}s before retry...")
            time.sleep(total_wait)


# Test it: a function that fails 3 times then succeeds
call_count = 0

def unreliable_api_call():
    global call_count
    call_count += 1
    if call_count < 4:
        raise ConnectionError(f"Server unavailable (attempt {call_count})")
    return {"status": "ok", "data": "agent response"}


random.seed(42)
call_count = 0
result = retry_with_backoff(unreliable_api_call)
print(f"\nSuccess: {result}")

---
## 6. Basic Logging

Once your agent starts making multiple tool calls in a loop, `print()` statements
become unmanageable. Python's built-in `logging` module gives you:

- **Log levels**: `DEBUG`, `INFO`, `WARNING`, `ERROR`, `CRITICAL` — filter noise
- **Timestamps**: know *when* each step happened
- **Consistent formatting**: structured output you can search and parse

| Level | When to use |
|-------|------------|
| `DEBUG` | Detailed diagnostic info (e.g., raw LLM response) |
| `INFO` | Normal operation milestones (e.g., "Tool call succeeded") |
| `WARNING` | Something unexpected but recoverable (e.g., "Retrying...") |
| `ERROR` | Something failed (e.g., "JSON parse error") |
| `CRITICAL` | The agent can't continue |

In [ ]:
import logging

# Configure logging for this notebook
# In notebooks, we configure the root logger; in production you'd use named loggers
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    force=True,  # reset any existing config
)

logger = logging.getLogger("agent")

# Demonstrate all log levels
logger.debug("Raw LLM response: '{\"action\": \"search\", \"query\": \"AI\"}'")
logger.info("Parsed action: search")
logger.warning("Tool 'search' took 3.2s (threshold: 2.0s)")
logger.error("Failed to parse LLM output as JSON")
logger.critical("No API key configured — agent cannot make LLM calls")

You can **change the log level** to filter out noise. Setting the level to `WARNING`
hides `DEBUG` and `INFO` messages:

In [ ]:
# Raise the log level — only WARNING and above will show
logger.setLevel(logging.WARNING)

logger.debug("This DEBUG message will be hidden")
logger.info("This INFO message will also be hidden")
logger.warning("This WARNING message is visible")
logger.error("This ERROR message is visible")

# Reset to DEBUG for the rest of the notebook
logger.setLevel(logging.DEBUG)

A practical pattern: use logging inside your error-handling code so you can trace
what happened without drowning in output.

In [ ]:
import json
import logging

logger = logging.getLogger("agent")

def execute_tool(tool_name: str, args: dict, tool_registry: dict) -> str:
    """Execute a tool by name with logging at every step."""
    logger.info(f"Executing tool: {tool_name} with args: {args}")
    
    if tool_name not in tool_registry:
        logger.error(f"Tool '{tool_name}' not found in registry")
        return f"Error: unknown tool '{tool_name}'"
    
    try:
        result = tool_registry[tool_name](args)
        logger.info(f"Tool '{tool_name}' returned: {result}")
        return result
    except Exception as e:
        logger.error(f"Tool '{tool_name}' raised {type(e).__name__}: {e}")
        return f"Error: {e}"


# Simple tool implementations
def mock_search(args: dict) -> str:
    query = args["query"]
    return f"Search results for '{query}': [result1, result2]"

def mock_calculate(args: dict) -> str:
    expression = args["expression"]
    return str(eval(expression))  # for demo only — never use eval in production!


registry = {"search": mock_search, "calculate": mock_calculate}

# Successful call
execute_tool("search", {"query": "AI agents"}, registry)

# Tool not found
execute_tool("summarize", {"text": "hello"}, registry)

# Tool raises an exception (missing key)
execute_tool("search", {}, registry)

---
## Putting It Together: `safe_json_parse`

Let's combine everything from this notebook into a realistic helper function that an
agent would actually use. `safe_json_parse` attempts to:

1. Parse the raw text as JSON
2. If that fails, try to extract JSON from the text using a regex (LLMs often wrap
   JSON in markdown code fences or add extra text)
3. Retry up to `max_retries` times with a provided callable
4. Log every attempt and outcome

In [ ]:
import json
import re
import logging
import time
import random

logger = logging.getLogger("agent")


def extract_json_from_text(text: str) -> str | None:
    """Try to extract a JSON object from text that may contain extra content.
    
    Handles common LLM patterns like:
      - JSON wrapped in ```json ... ``` code fences
      - JSON embedded in natural language text
    """
    # Try code fence extraction first
    fence_pattern = r"```(?:json)?\s*(\{.*?\})\s*```"
    match = re.search(fence_pattern, text, re.DOTALL)
    if match:
        return match.group(1)
    
    # Try to find a raw JSON object in the text
    brace_pattern = r"(\{[^{}]*\})"
    match = re.search(brace_pattern, text, re.DOTALL)
    if match:
        return match.group(1)
    
    return None


def safe_json_parse(
    text: str,
    max_retries: int = 3,
    get_new_text=None,  # callable that returns a new string to try
) -> dict | None:
    """Robustly parse JSON from text with retries and fallbacks.
    
    Strategy:
      1. Try direct JSON parse
      2. Try regex extraction + parse
      3. If get_new_text provided, retry with new text
    """
    for attempt in range(1, max_retries + 1):
        logger.info(f"Parse attempt {attempt}/{max_retries}")
        logger.debug(f"Input text: '{text[:80]}...'" if len(text) > 80 else f"Input text: '{text}'")
        
        # Strategy 1: direct parse
        try:
            result = json.loads(text)
            logger.info(f"Direct JSON parse succeeded on attempt {attempt}")
            return result
        except json.JSONDecodeError as e:
            logger.warning(f"Direct parse failed: {e.msg} at pos {e.pos}")
        
        # Strategy 2: regex extraction
        extracted = extract_json_from_text(text)
        if extracted:
            try:
                result = json.loads(extracted)
                logger.info(f"Regex extraction + parse succeeded on attempt {attempt}")
                return result
            except json.JSONDecodeError as e:
                logger.warning(f"Regex-extracted text also failed to parse: {e.msg}")
        else:
            logger.warning("No JSON-like content found via regex")
        
        # Strategy 3: get fresh text and retry
        if get_new_text and attempt < max_retries:
            logger.info("Requesting fresh text for retry...")
            text = get_new_text()
    
    logger.error(f"All {max_retries} parse attempts failed")
    return None

In [ ]:
# Test 1: Clean JSON — parses immediately
print("=" * 50)
print("Test 1: Clean JSON")
result = safe_json_parse('{"action": "search", "query": "agents"}')
print(f"Result: {result}")

# Test 2: JSON inside markdown code fence
print("\n" + "=" * 50)
print("Test 2: JSON in code fence")
messy_output = """Sure! Here's the action:
```json
{"action": "calculate", "expression": "2 + 2"}
```
Let me know if you need anything else."""
result = safe_json_parse(messy_output)
print(f"Result: {result}")

# Test 3: Completely broken text, but retry provides good text
print("\n" + "=" * 50)
print("Test 3: Broken text with retry")
retry_texts = iter([
    "Still not JSON...",
    '{"action": "search", "query": "fixed"}',
])
result = safe_json_parse(
    "This is total garbage, not JSON",
    max_retries=3,
    get_new_text=lambda: next(retry_texts),
)
print(f"Result: {result}")

---
## Try It Yourself

### Exercise 1: Handle Division by Zero

Write a function `safe_divide(a, b)` that:
- Returns `a / b` on success
- Returns `None` and prints an error message if `b` is zero
- Returns `None` and prints an error message if `a` or `b` aren't numbers

In [ ]:
# Exercise 1: Your code here
def safe_divide(a, b):
    """Safely divide a by b, handling errors gracefully."""
    pass  # Replace with your implementation


# Test cases (uncomment when ready):
# print(safe_divide(10, 3))       # Should print ~3.333
# print(safe_divide(10, 0))       # Should print error, return None
# print(safe_divide("ten", 3))    # Should print error, return None

### Exercise 2: `retry_with_backoff` Function

Write your own `retry_with_backoff(func, max_retries=3, base_delay=0.1)` that:
- Calls `func()` up to `max_retries` times
- On failure, waits `base_delay * (2 ** attempt)` seconds before retrying
- Returns the result on success
- Raises the last exception if all retries fail
- Prints the attempt number and wait time for each retry

Test it with a function that raises `ValueError` the first 2 times, then returns `"success"`.

In [ ]:
# Exercise 2: Your code here
import time

def my_retry_with_backoff(func, max_retries=3, base_delay=0.1):
    """Retry func with exponential backoff."""
    pass  # Replace with your implementation


# Test helper (uncomment when ready):
# fail_count = 0
# def flaky_function():
#     global fail_count
#     fail_count += 1
#     if fail_count < 3:
#         raise ValueError(f"Fail #{fail_count}")
#     return "success"
# 
# fail_count = 0
# result = my_retry_with_backoff(flaky_function)
# print(f"Got: {result}")

### Exercise 3: `safe_tool_call` Wrapper with Logging

Write a function `safe_tool_call(tool_name, tool_fn, args)` that:
- Uses `logging` (not `print`) to log an `INFO` message before calling the tool
- Calls `tool_fn(**args)` inside a `try`/`except`
- Logs the result at `INFO` level on success
- Logs the error at `ERROR` level on failure
- Returns a dict: `{"success": True, "result": ...}` or `{"success": False, "error": ...}`
- Uses `finally` to log a `DEBUG` message that the tool call is complete

In [ ]:
# Exercise 3: Your code here
import logging

logger = logging.getLogger("agent")

def safe_tool_call(tool_name: str, tool_fn, args: dict) -> dict:
    """Safely call a tool with logging."""
    pass  # Replace with your implementation


# Test tools (uncomment when ready):
# def add(x, y):
#     return x + y
#
# def broken_tool(text):
#     raise RuntimeError("Tool crashed!")
#
# print(safe_tool_call("add", add, {"x": 2, "y": 3}))
# print(safe_tool_call("broken", broken_tool, {"text": "hello"}))

### Exercise 4 (Stretch): Agent Step Executor

Combine everything: write a function `execute_agent_step(raw_llm_output, tool_registry)`
that:
1. Uses `safe_json_parse` (from the capstone above) to parse the LLM output
2. Validates that the parsed dict has an `"action"` key (raise `ValueError` if not)
3. Looks up the action in `tool_registry` (raise `KeyError` if not found)
4. Calls the tool with the parsed arguments
5. Returns the tool result
6. Logs every step
7. Handles all errors gracefully — never crashes, always returns a useful error message

In [ ]:
# Exercise 4: Your code here

def execute_agent_step(raw_llm_output: str, tool_registry: dict) -> dict:
    """Execute one step of an agent loop: parse -> validate -> call tool."""
    pass  # Replace with your implementation


# Test (uncomment when ready):
# def search(query):
#     return f"Results for '{query}'"
#
# registry = {"search": search}
#
# # Should succeed:
# print(execute_agent_step('{"action": "search", "query": "AI"}', registry))
#
# # Should handle bad JSON:
# print(execute_agent_step('not json', registry))
#
# # Should handle missing action:
# print(execute_agent_step('{"query": "AI"}', registry))
#
# # Should handle unknown tool:
# print(execute_agent_step('{"action": "fly", "dest": "moon"}', registry))

---

## Summary

You now have the error-handling foundations that every agent needs:

| Concept | Why it matters for agents |
|---------|-------------------------|
| `try`/`except` | Catch failures without crashing the agent loop |
| Specific exceptions | Handle different failure modes differently |
| `else`/`finally` | Clean separation of success path, error path, and cleanup |
| `raise` + custom exceptions | Signal tool errors and validation failures clearly |
| Retry with backoff | Survive rate limits and transient API failures |
| `logging` | Debug what happened on step 47 of a 50-step plan |

**Next up:** [06. HTTP and APIs](06_http_and_apis.ipynb) — making actual API calls to LLM
providers, where all of these error-handling patterns become essential.